## Notebook 04 - Player Centralities per match
Núria Pascual Salas

**Content:** Calculates PageRank and betweenness centrality for all players in each match, across the 20 teams. The resulting player-level dataset (one row per match, team, and player) serves as the basis for the later analyses. No minutes or position filters have been applied yet.

**Outputs:**
- outputs/csv/player_centralities_per_match.csv (master player-level CSV: match/team
  identifiers, outcome, player, rank, pagerank, betweenness, minutes)

**Used in:** Chapter 5 (player-level analysis) and notebook 06.

In [1]:
from utils import *
import networkx as nx
import numpy as np
import pandas as pd
from collections import defaultdict
import os
import time

CSV_DIR = 'outputs/csv'
os.makedirs(CSV_DIR, exist_ok=True)

### 1. Helper functions: minutes played and match metadata

In [2]:
def get_minutes_played(events, team_name):
    """
    Compute minutes played by each player in a match.
    
    Uses 'Starting XI' and 'Substitution' events to track when players enter and leave.
    Returns dict {player_name: minutes_played}.
    """
    # Find max minute reached in this match (handles extra time)
    max_minute = max((e.get('minute', 0) for e in events), default=90)
    if max_minute < 90:
        max_minute = 90
    
    # Find starting XI for this team
    starting_xi = []
    for e in events:
        if (e.get('type', {}).get('name') == 'Starting XI'
            and e.get('team', {}).get('name') == team_name):
            lineup = e.get('tactics', {}).get('lineup', [])
            starting_xi = [p['player']['name'] for p in lineup if 'player' in p]
            break
    
    # Each starter begins at minute 0
    entry_time = {p: 0 for p in starting_xi}
    exit_time  = {}
    
    # Process substitutions
    for e in events:
        if (e.get('type', {}).get('name') == 'Substitution'
            and e.get('team', {}).get('name') == team_name):
            minute_sub = e.get('minute', max_minute)
            player_out = e.get('player', {}).get('name')
            player_in  = e.get('substitution', {}).get('replacement', {}).get('name')
            
            if player_out:
                exit_time[player_out] = minute_sub
            if player_in:
                entry_time[player_in] = minute_sub
    
    # Compute minutes for each player who appears
    minutes = {}
    for player in entry_time:
        start = entry_time[player]
        end   = exit_time.get(player, max_minute)
        minutes[player] = end - start
    
    return minutes


def get_match_metadata(events, team_id):
    """Get the home/away teams, score and outcome for the given team."""
    home_team, away_team = None, None
    team_is_home = False
    
    # Find team names from Starting XI events
    teams_seen = []
    for e in events:
        if e.get('type', {}).get('name') == 'Starting XI':
            team_name = e.get('team', {}).get('name')
            tid = e.get('team', {}).get('id')
            if (team_name, tid) not in teams_seen:
                teams_seen.append((team_name, tid))
    
    if len(teams_seen) != 2:
        return None, None, False, '?-?', 'unknown'
    
    # The first team in events is the home team
    home_team, home_id = teams_seen[0]
    away_team, away_id = teams_seen[1]
    team_is_home = (team_id == home_id)
    
    # Count goals
    goals_home = sum(
        1 for e in events
        if (e.get('type', {}).get('name') == 'Shot'
            and e.get('shot', {}).get('outcome', {}).get('name') == 'Goal'
            and e.get('team', {}).get('id') == home_id)
    )
    goals_away = sum(
        1 for e in events
        if (e.get('type', {}).get('name') == 'Shot'
            and e.get('shot', {}).get('outcome', {}).get('name') == 'Goal'
            and e.get('team', {}).get('id') == away_id)
    )
    
    # Add own goals for the opposite team
    for e in events:
        if e.get('type', {}).get('name') == 'Own Goal For':
            scorer_team_id = e.get('team', {}).get('id')
            if scorer_team_id == home_id:
                goals_home += 1
            elif scorer_team_id == away_id:
                goals_away += 1
    
    score = f"{goals_home}-{goals_away}"
    
    # Determine outcome for the target team
    if team_is_home:
        if   goals_home > goals_away: outcome = 'win'
        elif goals_home < goals_away: outcome = 'loss'
        else:                          outcome = 'draw'
    else:
        if   goals_away > goals_home: outcome = 'win'
        elif goals_away < goals_home: outcome = 'loss'
        else:                          outcome = 'draw'
    
    return home_team, away_team, team_is_home, score, outcome

### 2. Compute per-match player centralities

In [3]:
all_rows = []
start_time = time.time()

n_matches_processed = 0
n_team_matches = 0

for m_id, events in stream_matches_from_zip(zip_path, folder_laliga, "_events.json"):
    n_matches_processed += 1
    
    # Identify the two teams in this match
    teams_in_match = list({(e['team']['id'], e['team']['name'])
                           for e in events if 'team' in e})
    
    for team_id, team_name in teams_in_match:
        if team_id not in all_teams:
            continue
        
        # Match metadata (same for all players of this team in this match)
        home_team, away_team, team_is_home, score, outcome = get_match_metadata(
            events, team_id
        )
        
        # Build the network and compute centralities
        G = build_passing_network(events, team_name)
        if G.number_of_nodes() == 0:
            continue
        
        pr = compute_pagerank_match(G)
        bt = compute_betweenness_match(G)
        minutes = get_minutes_played(events, team_name)
        
        # Sort players by PageRank descending
        sorted_players = sorted(pr.items(), key=lambda x: x[1], reverse=True)
        
        for rank_idx, (player_name, pr_value) in enumerate(sorted_players, 1):
            all_rows.append({
                'match_id':        m_id,
                'home_team':       home_team,
                'away_team':       away_team,
                'team_id':         team_id,
                'team_name':       team_name,
                'team_is_home':    team_is_home,
                'score':           score,
                'outcome':         outcome,
                'player_name':     player_name,
                'rank':            rank_idx,
                'pagerank':        pr_value,
                'pagerank_pct':    round(pr_value * 100, 2),
                'betweenness':     round(bt.get(player_name, 0), 4),
                'minutes_played':  minutes.get(player_name, 0),
            })
        
        n_team_matches += 1
    
    if n_matches_processed % 50 == 0:
        elapsed = time.time() - start_time
        print(f"  Processed {n_matches_processed} matches | "
              f"{n_team_matches} team-match pairs | "
              f"{len(all_rows)} player rows | "
              f"elapsed {elapsed:.1f}s")

print(f"\nDone.")
print(f"  Total matches processed: {n_matches_processed}")
print(f"  Total team-match pairs:  {n_team_matches}")
print(f"  Total player rows:       {len(all_rows)}")
print(f"  Total time:              {time.time() - start_time:.1f}s")

  Processed 50 matches | 100 team-match pairs | 1545 player rows | elapsed 6.5s
  Processed 100 matches | 200 team-match pairs | 3091 player rows | elapsed 13.2s
  Processed 150 matches | 300 team-match pairs | 4650 player rows | elapsed 20.2s
  Processed 200 matches | 400 team-match pairs | 6204 player rows | elapsed 27.3s
  Processed 250 matches | 500 team-match pairs | 7761 player rows | elapsed 35.1s
  Processed 300 matches | 600 team-match pairs | 9322 player rows | elapsed 42.6s
  Processed 350 matches | 700 team-match pairs | 10859 player rows | elapsed 50.2s

Done.
  Total matches processed: 380
  Total team-match pairs:  760
  Total player rows:       11790
  Total time:              54.7s


### 3. Build the DataFrame and check

In [4]:
df_master = pd.DataFrame(all_rows)

print(f"Total rows: {len(df_master)}")
print(f"Total unique matches: {df_master['match_id'].nunique()}")
print(f"Total unique teams: {df_master['team_id'].nunique()}")
print(f"Total unique players: {df_master['player_name'].nunique()}")
print()
print("Distribution of outcomes (rows, not matches):")
print(df_master['outcome'].value_counts())
print()
print("First 5 rows:")
print(df_master.head().to_string(index=False))

Total rows: 11790
Total unique matches: 380
Total unique teams: 20
Total unique players: 596

Distribution of outcomes (rows, not matches):
outcome
loss    4274
win     4224
draw    3292
Name: count, dtype: int64

First 5 rows:
match_id        home_team away_team  team_id        team_name  team_is_home score outcome               player_name  rank  pagerank  pagerank_pct  betweenness  minutes_played
 3894664 Deportivo Alavés   Granada      206 Deportivo Alavés          True   3-1     win Luis Jesús Rioja González     1  0.134968         13.50       0.3379              89
 3894664 Deportivo Alavés   Granada      206 Deportivo Alavés          True   3-1     win     Javier López Carballo     2  0.100009         10.00       0.2198              95
 3894664 Deportivo Alavés   Granada      206 Deportivo Alavés          True   3-1     win        Ander Guevara Lajo     3  0.094683          9.47       0.1703              68
 3894664 Deportivo Alavés   Granada      206 Deportivo Alavés          T

In [8]:
# Check 1: Each team should have 38 matches
matches_per_team = df_master.groupby('team_name')['match_id'].nunique().sort_values()
print("Matches per team:")
print(matches_per_team.to_string())
print()

# Check 2: Each team should have a roughly balanced distribution of outcomes
team_outcomes = df_master.drop_duplicates(['match_id', 'team_id']).groupby(
    ['team_name', 'outcome']).size().unstack(fill_value=0)
print("\nOutcomes per team (unique matches):")
print(team_outcomes.to_string())

# Check 3: Barcelona should have 26 wins, 7 draws and 5 losses
barca_outcomes = df_master[df_master['team_name'] == 'Barcelona'].drop_duplicates(
    ['match_id'])['outcome'].value_counts()
print(f"\nFC Barcelona outcomes (sanity check):")
print(barca_outcomes.to_string())
print("Expected: 26 wins, 7 draws and 5 losses")

Matches per team:
team_name
Almería             38
Sevilla             38
Real Sociedad       38
Real Madrid         38
Real Betis          38
Rayo Vallecano      38
Osasuna             38
Mallorca            38
Las Palmas          38
Granada             38
Girona              38
Getafe              38
Deportivo Alavés    38
Cádiz               38
Celta Vigo          38
Barcelona           38
Atlético Madrid     38
Athletic Club       38
Valencia            38
Villarreal          38


Outcomes per team (unique matches):
outcome           draw  loss  win
team_name                        
Almería             12    23    3
Athletic Club       11     8   19
Atlético Madrid      4    10   24
Barcelona            7     5   26
Celta Vigo          11    17   10
Cádiz               15    17    6
Deportivo Alavés    10    16   12
Getafe              13    15   10
Girona               6     7   25
Granada              9    25    4
Las Palmas          10    18   10
Mallorca            16    14    

### 4. Save the player-level CSV

In [6]:
output_path = f'{CSV_DIR}/player_centralities_per_match.csv'
df_master.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"Saved master CSV to: {output_path}")
print(f"File size: {os.path.getsize(output_path) / 1024:.1f} KB")
print(f"Total rows: {len(df_master)}")

Saved master CSV to: outputs/csv/player_centralities_per_match.csv
File size: 1368.5 KB
Total rows: 11790


In [7]:
import pandas as pd
df = pd.read_csv(f'{CSV_DIR}/player_centralities_per_match.csv')

# Outcomes per equip
team_outcomes = df.drop_duplicates(['match_id', 'team_id']).groupby(
    ['team_name', 'outcome']).size().unstack(fill_value=0)
print(team_outcomes)

outcome           draw  loss  win
team_name                        
Almería             12    23    3
Athletic Club       11     8   19
Atlético Madrid      4    10   24
Barcelona            7     5   26
Celta Vigo          11    17   10
Cádiz               15    17    6
Deportivo Alavés    10    16   12
Getafe              13    15   10
Girona               6     7   25
Granada              9    25    4
Las Palmas          10    18   10
Mallorca            16    14    8
Osasuna              9    17   12
Rayo Vallecano      14    16    8
Real Betis          15     9   14
Real Madrid          8     1   29
Real Sociedad       12    10   16
Sevilla             11    17   10
Valencia            10    15   13
Villarreal          11    13   14
